# Kernel PCA view of the K-means clusters on DGCI active dimensions

In [ ]:
# interactive, rotatable 3D figures (needs: pip install ipympl)
%matplotlib widget

import numpy as np                                          # array maths
import matplotlib.pyplot as plt                             # plotting
from sklearn.cluster import KMeans                          # the clustering used in the paper
from sklearn.metrics import silhouette_score                # cluster separation without labels

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/unregistered_128_2M_4k/latent_codes.npy"
enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 for stable distances

In [ ]:
dim_median   = np.median(enc_output, axis=0)                # typical value per dimension
dim_mad      = np.median(np.abs(enc_output - dim_median), axis=0)  # robust spread per dimension
active_dims  = np.where(dim_mad > 3e-4)[0]                  # threshold sits in the gap of the MAD plot

codes_active = enc_output[:, active_dims]                   # (419, 46)
print(codes_active.shape)                                   # confirm the count

In [ ]:
n_clusters = 4                                              # K chosen in kmeans_on_active.ipynb

# same settings as kmeans_on_active.ipynb, so the labels are identical
kmeans      = KMeans(n_clusters=n_clusters, n_init=20, random_state=0)  # n_init = restarts, best kept
cluster_ids = kmeans.fit_predict(codes_active)              # cluster id for each of the 419 shapes
print(np.bincount(cluster_ids))                             # shapes per cluster, expect [77 114 141 87]

## Pipeline

```
          DGCI latent codes (419 x 128)
                     │
                     ▼
        Active dimensions (419 x 46)
                     │
          ┌──────────┴──────────┐
          │                     │
          ▼                     ▼
       K-means             Kernel PCA (RBF)
          │                     │
          ▼                     ▼
    cluster labels        3D coordinates
          │                     │
          └──────────┬──────────┘
                     ▼
     3D plot: points placed by kernel PCA,
     colored by K-means label
```

Kernel PCA never sees the labels; they are only used for coloring and for the silhouette score.
The same recipe is applied to a Gaussian cloud with no clusters, as a baseline.

In [ ]:
def plot_clusters_3d(xyz, labels, title):
    """3D scatter colored by cluster, cropped to the 1st-99th percentile."""
    fig = plt.figure(figsize=(8, 7))                        # one large panel
    ax = fig.add_subplot(projection="3d")                   # 3D axes
    for c in range(n_clusters):                             # one call per cluster gives a legend
        pts = xyz[labels == c]                              # points in cluster c
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=14, alpha=0.7, label=f"cluster {c}")
    low = np.percentile(xyz, 1, axis=0)                     # crop far outliers
    high = np.percentile(xyz, 99, axis=0)                   # upper bound per axis
    ax.set_xlim(low[0], high[0])                            # axis 1 range
    ax.set_ylim(low[1], high[1])                            # axis 2 range
    ax.set_zlim(low[2], high[2])                            # axis 3 range
    ax.set_title(title)                                     # which view this is
    ax.legend()                                             # cluster names
    return ax                                               # lets the caller change the angle


# baseline: same mean and covariance as your codes, but no clusters
rng = np.random.default_rng(0)                              # fixed seed, reproducible baseline
data_mean = codes_active.mean(axis=0)                       # same center as your codes
data_cov = np.cov(codes_active, rowvar=False)               # same covariance (rowvar=False: columns = dims)
fake = rng.multivariate_normal(data_mean, data_cov, size=len(codes_active))  # structureless copy
fake_kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=0)  # identical K-means settings
fake_ids = fake_kmeans.fit_predict(fake)                    # K-means always returns 4 groups

In [ ]:
from sklearn.decomposition import KernelPCA                 # PCA on kernel similarities
from sklearn.metrics import pairwise_distances              # all pairwise shape distances


def rbf_gamma(data):
    """Median rule: gamma such that a typical pair has similarity ~0.37."""
    sq_dists = pairwise_distances(data) ** 2                # (419, 419) squared distances
    typical = np.median(sq_dists[sq_dists > 0])             # typical pair, skipping self-distances
    return 1.0 / typical                                    # exp(-1) ≈ 0.37 for a typical pair


def kernel_pca_3d(data, gamma):
    """Top 3 kernel PCA coordinates and the share of kernel variance they hold."""
    kpca = KernelPCA(kernel="rbf", gamma=gamma)             # keep all components to measure the share
    embedding = kpca.fit_transform(data)                    # (419, n_components), sorted by eigenvalue
    top3_share = kpca.eigenvalues_[:3].sum() / kpca.eigenvalues_.sum()  # like explained_variance_ratio_
    return embedding[:, :3], top3_share                     # 3 plot axes + how much they show


base_gamma = rbf_gamma(codes_active)                        # data-driven width for this scale
default_gamma = 1 / codes_active.shape[1]                   # what sklearn would use (1 / n_features)
print(f"median-rule gamma: {base_gamma:.3g}  (sklearn default would be {default_gamma:.3g})")

# fixed grid chosen in advance: narrower, standard, wider (report all, don't pick the prettiest)
for factor in [0.01, 0.025, 0.05, 0.1, 0.5, 1, 10]:
    gamma = factor * base_gamma                              # kernel width for this run
    real_3d, real_share = kernel_pca_3d(codes_active, gamma)  # the K-means data
    fake_3d, _ = kernel_pca_3d(fake, factor * rbf_gamma(fake))  # same recipe on the no-cluster baseline
    real_sil = silhouette_score(real_3d, cluster_ids)        # separation visible in the 3D view
    fake_sil = silhouette_score(fake_3d, fake_ids)           # what "no structure" gives in that view
    print(f"gamma x{factor:<4}: top-3 share {real_share:.1%}, "  # how much the view shows
          f"3D silhouette real {real_sil:.3f}, baseline {fake_sil:.3f}")  # real vs no-cluster
    plot_clusters_3d(real_3d, cluster_ids, f"Kernel PCA (RBF), gamma = {factor} x median rule")
    plt.show()                                              # one figure per gamma

In [ ]:
# optional check: do the near-constant dims change the picture?
all_3d, _ = kernel_pca_3d(enc_output, rbf_gamma(enc_output))  # all 128 dims, own median-rule gamma
act_3d, _ = kernel_pca_3d(codes_active, base_gamma)           # active dims, as above
all_sil = silhouette_score(all_3d, cluster_ids)               # separation using all dims
act_sil = silhouette_score(act_3d, cluster_ids)               # separation using active dims
print(f"3D silhouette  all dims {all_sil:.3f}, active dims {act_sil:.3f}")  # similar = harmless